In [123]:
import pandas as pd 

In [239]:
df1 = pd.read_csv("/Users/victoriacano/Downloads/df_final_web_data_pt_1.txt", sep=",")
df2 = pd.read_csv("/Users/victoriacano/Downloads/df_final_web_data_pt_2.txt", sep=",")

df1
df2

,client_id,visitor_id,visit_id,process_step,date_time
0,763412,601952081_10457207388,397475557_40440946728_419634,confirm,2017-06-06 08:56:00
1,6019349,442094451_91531546617,154620534_35331068705_522317,confirm,2017-06-01 11:59:27
2,6019349,442094451_91531546617,154620534_35331068705_522317,step_3,2017-06-01 11:58:48
3,6019349,442094451_91531546617,154620534_35331068705_522317,step_2,2017-06-01 11:58:08
4,6019349,442094451_91531546617,154620534_35331068705_522317,step_1,2017-06-01 11:57:58
...,...,...,...,...,...
412259,9668240,388766751_9038881013,922267647_3096648104_968866,start,2017-05-24 18:46:10
412260,9668240,388766751_9038881013,922267647_3096648104_968866,start,2017-05-24 18:45:29
412261,9668240,388766751_9038881013,922267647_3096648104_968866,step_1,2017-05-24 18:44:51
412262,9668240,388766751_9038881013,922267647_3096648104_968866,start,2017-05-24 18:44:34


In [240]:
df_web_data = pd.concat([df1, df2], axis=0, join="outer")

In [241]:
df_web_data["client_id"] = pd.to_numeric(df_web_data["client_id"], errors='coerce', downcast='integer')
df_web_data["client_id"] = df_web_data["client_id"].astype("Int64")
df_web_data["date_time"] = pd.to_datetime(df_web_data["date_time"])

In [242]:
df_web_data.dtypes

client_id                Int64
visitor_id                 str
visit_id                   str
process_step               str
date_time       datetime64[us]
dtype: object

In [243]:
#Checking for null values 
df_web_data.isnull().values.any()


np.False_

In [244]:
df_web_data["client_id"].isnull().sum()

np.int64(0)

In [245]:
df_web_data["visitor_id"].isnull().sum()

np.int64(0)

In [246]:
df_web_data["visit_id"].isnull().sum()

np.int64(0)

In [247]:
df_web_data["process_step"].isnull().sum()

np.int64(0)

In [248]:
df_web_data["date_time"].isnull().sum()

np.int64(0)

In [249]:
#Check for duplicates:

df_web_data_duplicates = df_web_data[df_web_data.duplicated('client_id')]
df_web_data_duplicates

#When checking for client_id duplicates many appear but the date_time and process are different because the same client may have tried to carry out a step again 

,client_id,visitor_id,visit_id,process_step,date_time
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22
3,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13
4,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04
5,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:17:15
...,...,...,...,...,...
412258,9668240,388766751_9038881013,922267647_3096648104_968866,step_1,2017-05-24 18:48:02
412259,9668240,388766751_9038881013,922267647_3096648104_968866,start,2017-05-24 18:46:10
412260,9668240,388766751_9038881013,922267647_3096648104_968866,start,2017-05-24 18:45:29
412261,9668240,388766751_9038881013,922267647_3096648104_968866,step_1,2017-05-24 18:44:51


In [250]:
#Total number of clients in the dataset (clients that did the process)

total_clients = df_web_data["client_id"].nunique()
print(f"Total number of clients in the dataset: {total_clients}")

Total number of clients in the dataset: 120157


In [251]:
#Checking how many clients redid confirmed step to only keep the last one as the end of the process 

confirmed = df_web_data[df_web_data["process_step"] == "confirm"]
counts = confirmed.groupby("client_id").size()
print(counts)

counts.sort_values(ascending=False)

client_id
169        1
546        1
555        1
647        1
722        1
          ..
9999451    1
9999729    1
9999768    1
9999839    1
9999875    1
Length: 81145, dtype: int64


client_id
7347761    24
8167818    23
8374531    20
4127054    20
244312     19
           ..
3648217     1
3647935     1
3647814     1
3647758     1
9999875     1
Length: 81145, dtype: int64

In [254]:
#Keeping only the last confirm process for the clients that did it more than once

df_web_data = df_web_data.sort_values(['client_id', 'date_time']) \
                     .groupby('client_id') \
                     .tail(1)

df_web_data.to_csv("data/clean_web_data.csv", index=False)